In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import random
def set_time(tahun, bulan, hari, jam, menit, detik):
    return datetime(tahun, bulan, hari, jam, menit, detik)

start_project = set_time(2021, 8, 3, 7, 10, 30)

## Create Master Data

## Employee Data

In [ ]:

def create_employees_df():
    return pd.DataFrame({
        "employee_id": pd.Series(dtype="object"),
        "employee_name": pd.Series(dtype="object"),
        "role": pd.Series(dtype="object"),
        "city": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),
        "start_date": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def get_next_sequence(df):
    if df.empty:
        return 1

    return df.shape[0]

def create_code_id(df, prefix, column_name, num=1):
    if df.empty:
        sequence = get_next_sequence(df)
    else :
        sequence = get_next_sequence(df) +1
    num_list = []
    for _ in range(num):

        rand_4 = np.random.randint(1000, 9999)
        rand_3 = np.random.randint(100, 999)

        new_id = f"{prefix}-{rand_4}-{rand_3}-{sequence:03}"

        if df.empty or new_id not in df[column_name].values:
            if num == 1:
                return new_id
                break
            sequence += 1
            num_list.append(new_id)
    if num > 1:
        return num_list

def add_employee(df, data, time):
    required_fields = ["employee_name", "role", "city"]

    # Validasi field
    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    # ===== DETECT MULTI / SINGLE =====
    is_multiple = isinstance(data["employee_name"], list)

    if is_multiple:
        new_df = pd.DataFrame(data)
        n = len(new_df)
    else:
        new_df = pd.DataFrame([data])
        n = 1

    # ===== AUTO FIELD =====
    new_df["start_date"] = time
    new_df["created_at"] = time
    new_df["updated_at"] = time
    new_df["status"] = "Active"

    # ===== GENERATE ID =====
    new_ids = create_code_id(df, "EMP", "employee_id", n)
    if n == 1:
        new_df["employee_id"] = new_ids
    else:
        new_df["employee_id"] = new_ids

    # ===== CREATED BY =====
    if df.empty:
        new_df["created_by"] = new_df["employee_id"]
    else:
        admin_df = df[df["role"] == "admin"]
        creator = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"
        new_df["created_by"] = creator

    # ===== INSERT =====
    df = pd.concat([df, new_df], ignore_index=True)

    return df

def update_employee(df, employee_id, update_data, time):
    idx = df[df["employee_id"] == employee_id].index

    if len(idx) == 0:
        raise ValueError("Employee tidak ditemukan")

    # Update field
    for key, value in update_data.items():
        if key in df.columns and key != "employee_id":
            df.loc[idx, key] = value

    # Update timestamp
    df.loc[idx, "updated_at"] = time

    return df

def deactivate_employee(df, employee_id, reason="resigned"):
    idx = df[df["employee_id"] == employee_id].index

    if len(idx) == 0:
        raise ValueError("Employee tidak ditemukan")

    df.loc[idx, "status"] = "inactive"
    df.loc[idx, "updated_at"] = start_project

    # optional: simpan alasan di kolom explanation kalau ada
    if "explanation" in df.columns:
        df.loc[idx, "explanation"] = reason

    return df

In [ ]:
#create master data
employees_df = create_employees_df()

data = {
    "employee_name": "Fahmi",
    "role": "admin",
    "city": "Jakarta"
}
employees_df = add_employee(employees_df, data, start_project)
employees_df

/tmp/ipykernel_642/2259693139.py:82: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_df], ignore_index=True)


,employee_id,employee_name,role,city,status,start_date,created_at,updated_at,created_by
0,EMP-1998-144-001,Fahmi,admin,Jakarta,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


In [ ]:
data = {
    "employee_name": ["Ahmad", "Farhan"],
    "role": ["qc_packing", "warehouse"],
    "city": ["Tangerang", "Tangerang"],
}
employees_df = add_employee(employees_df, data, start_project)
employees_df

,employee_id,employee_name,role,city,status,start_date,created_at,updated_at,created_by
0,EMP-1998-144-001,Fahmi,admin,Jakarta,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,EMP-9316-263-002,Ahmad,qc_packing,Tangerang,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,EMP-4124-636-003,Farhan,warehouse,Tangerang,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


## Suppliers Data

In [ ]:
def create_suppliers_df():
    return pd.DataFrame({
        "supplier_id": pd.Series(dtype="object"),
        "supplier_name": pd.Series(dtype="object"),
        "city": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def add_supplier(df, data, time):
    required_fields = ["supplier_name", "city"]

    # ===== VALIDASI =====
    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    # ===== DETECT MULTI / SINGLE =====
    is_multiple = isinstance(data["supplier_name"], list)

    if is_multiple:
        new_df = pd.DataFrame(data)
        n = len(new_df)
    else:
        new_df = pd.DataFrame([data])
        n = 1

    # ===== AUTO FIELD =====
    new_df["created_at"] = time
    new_df["updated_at"] = time
    new_df["status"] = "Active"

    # ===== GENERATE ID =====
    ids = create_code_id(df, "SPLR", "supplier_id", n)

    if n == 1:
        new_df["supplier_id"] = ids
    else:
        new_df["supplier_id"] = ids

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    creator = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"
    new_df["created_by"] = creator

    # ===== INSERT =====
    df = pd.concat([df, new_df], ignore_index=True)

    return df

def update_supplier(df, supplier_id, update_data, time):
    idx = df[df["supplier_id"] == supplier_id].index

    if len(idx) == 0:
        raise ValueError("Supplier tidak ditemukan")

    for key, value in update_data.items():
        if key in df.columns and key != "supplier_id":
            df.loc[idx, key] = value

    df.loc[idx, "updated_at"] = time

    return df

def deactivate_supplier(df, supplier_id, time):
    idx = df[df["supplier_id"] == supplier_id].index

    if len(idx) == 0:
        raise ValueError("Supplier tidak ditemukan")

    df.loc[idx, "active"] = False
    df.loc[idx, "updated_at"] = time

    return df

In [ ]:
suppliers_df = create_suppliers_df()

#create master data
data_suppliers = {
    "supplier_name" : ["PT Bogo Nusantara Safety", "PT Optik Visor Jaya", "PT Syar'i Ride Apparel", "PT Motor Agung Laksana"],
    "city": ["Jakarta", "Tangerang", "Jakarta", "Tangerang"]
}
suppliers_df = add_supplier(suppliers_df, data_suppliers, start_project)
suppliers_df

/tmp/ipykernel_642/3447336789.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_df], ignore_index=True)


,supplier_id,supplier_name,city,status,created_at,updated_at,created_by
0,SPLR-6274-501-001,PT Bogo Nusantara Safety,Jakarta,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,SPLR-6208-341-002,PT Optik Visor Jaya,Tangerang,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,SPLR-9649-612-003,PT Syar'i Ride Apparel,Jakarta,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
3,SPLR-6297-996-004,PT Motor Agung Laksana,Tangerang,Active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


## Products Data

In [ ]:
def create_products_df():
    return pd.DataFrame({
        "product_id": pd.Series(dtype="object"),
        "product_name": pd.Series(dtype="object"),
        "category": pd.Series(dtype="object"),
        "variant": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),
        "vendor_id": pd.Series(dtype="object"),
        "cost_price": pd.Series(dtype="float"),
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })
def calculate_price(cost):
    selling_price = cost * (1 + 0.4)   # +40%
    tax = selling_price * 0.12         # pajak 12%
    return selling_price, tax

def add_product(df, data, time):

    # requirement fields
    required_fields = ["product_name", "category", "variant", "vendor_id", "cost_price"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    # ===== DETECT MULTI / SINGLE =====
    is_multiple = isinstance(data["product_name"], list)

    if is_multiple:
        new_df = pd.DataFrame(data)
        n = len(new_df)
    else:
        new_df = pd.DataFrame([data])
        n = 1

    new_df["created_at"] = time
    new_df["updated_at"] = time
    new_df["status"] = "active"

    # ===== GENERATE ID =====

    ids = create_code_id(df, "PRD", "product_id", n)
    new_df["product_id"] = ids

    # ===== CREATED BY =====

    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"

    new_df["created_by"] = admin_id

    # ===== INSERT =====
    df = pd.concat([df, new_df], ignore_index=True)

    return df

def update_product(df, product_id, update_data, time):
    idx = df[df["product_id"] == product_id].index

    if len(idx) == 0:
        raise ValueError("Product tidak ditemukan")

    for key, value in update_data.items():
        if key in df.columns and key != "product_id":
            df.loc[idx, key] = value

    # update harga kalau cost berubah
    if "cost_price" in update_data:
        cost = update_data["cost_price"]
        selling_price, tax = calculate_price(cost)
        df.loc[idx, "selling_price"] = selling_price
        df.loc[idx, "tax"] = tax

    df.loc[idx, "updated_at"] = time

    return df

def deactivate_product(df, product_id, time):
    idx = df[df["product_id"] == product_id].index

    if len(idx) == 0:
        raise ValueError("Product tidak ditemukan")

    df.loc[idx, "status"] = "inactive"
    df.loc[idx, "updated_at"] = time

    return df


In [ ]:
bogo_colors = [
    "grey gloss","grey doff","hitam gloss","mint","sage","hitam doff",
    "bluesky","kuning","cream","army doff","maroon doff","maroon gloss",
    "putih gloss","silver","darkgrey gloss","pink soft","capuchino"
]

kaca_colors = [
    "flat smoke","cembung smoke","flat clear","flat rainbow","pet doff","pet gloss"
]

hijab_colors = [
    "grey gloss","grey doff","hitam gloss","mint","sage","hitam doff",
    "bluesky","kuning","cream","army doff","maroon doff","maroon gloss",
    "putih gloss","darkgrey gloss","pink soft","capuchino"
]

cakil_colors = [
    "hitam gloss","hitam dof","army doff","putih gloss","darkgrey gloss","pink soft"
]

varian_products = {
    "helm bogo": bogo_colors,
    "accessories": kaca_colors,
    "helm hijab bogo": hijab_colors,
    "helm cakil": cakil_colors
}

vendor = {
    "helm bogo": suppliers_df.loc[0, "supplier_id"],
    "accessories": suppliers_df.loc[1, "supplier_id"],
    "helm hijab bogo": suppliers_df.loc[2, "supplier_id"],
    "helm cakil": suppliers_df.loc[3, "supplier_id"]
}

price = {
    "helm bogo": 193000,
    "accessories": 24000,
    "helm hijab bogo": 202000,
    "helm cakil": 191000
}

products_df = create_products_df()

for variant, colors in varian_products.items():
    for color in colors:
        data = {
            "product_name": f"{variant} {color}",
            "category": variant,
            "variant": color,
            "vendor_id": vendor[variant],
            "cost_price": price[variant]
        }

        products_df = add_product(products_df, data, start_project)
products_df


/tmp/ipykernel_642/3481554606.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_df], ignore_index=True)


,product_id,product_name,category,variant,status,vendor_id,cost_price,created_at,updated_at,created_by
0,PRD-8874-102-001,helm bogo grey gloss,helm bogo,grey gloss,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,PRD-5383-617-002,helm bogo grey doff,helm bogo,grey doff,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,PRD-2684-647-003,helm bogo hitam gloss,helm bogo,hitam gloss,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
3,PRD-9499-652-004,helm bogo mint,helm bogo,mint,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
4,PRD-7442-689-005,helm bogo sage,helm bogo,sage,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
5,PRD-7478-588-006,helm bogo hitam doff,helm bogo,hitam doff,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
6,PRD-4480-984-007,helm bogo bluesky,helm bogo,bluesky,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
7,PRD-3442-197-008,helm bogo kuning,helm bogo,kuning,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
8,PRD-6679-849-009,helm bogo cream,helm bogo,cream,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
9,PRD-2992-160-010,helm bogo army doff,helm bogo,army doff,active,SPLR-6274-501-001,193000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


## catalog product & catalog items

In [ ]:
def create_catalog_products_df():
    return pd.DataFrame({
        "catalog_product_id": pd.Series(dtype="object"),
        "product_name": pd.Series(dtype="object"),
        "is_bundle": pd.Series(dtype="bool"),
        "cost_price" : pd.Series(dtype="float"),
        "selling_price": pd.Series(dtype="float"),
        "tax": pd.Series(dtype="float"),
        "status": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_catalog_items_df():
    return pd.DataFrame({
        "catalog_product_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "quantity": pd.Series(dtype="int"),
    })

def add_catalog_product(catalog_df, items_df, data, time):
    required_fields = ["product_name", "is_bundle", "cost_price", "items"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    selling_price, tax = calculate_price(data['cost_price'])

    # ===== CREATE CATALOG PRODUCT =====
    new_catalog = pd.DataFrame([{
        "product_name": data["product_name"],
        "is_bundle": data["is_bundle"],
        "cost_price": data["cost_price"],
        "selling_price": selling_price,
        "tax": tax,
        "status": "active",
        "created_at": time,
        "updated_at": time,
    }])

    # generate ID
    catalog_id = create_code_id(catalog_df, "CAT", "catalog_product_id")
    new_catalog["catalog_product_id"] = catalog_id

    # created_by
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"
    new_catalog["created_by"] = admin_id

    # ===== VALIDASI ITEMS 🔥 =====
    items = data["items"]

    if not isinstance(items, list) or len(items) == 0:
        raise ValueError("items harus list dan tidak boleh kosong")

    if not any(item.get("is_primary", False) for item in items):
        raise ValueError("Minimal harus ada 1 primary product")

    # ===== CREATE ITEMS =====
    items_list = []

    for item in items:
        if "product_id" not in item or "quantity" not in item:
            raise ValueError("item harus punya product_id dan quantity")

        items_list.append({
            "catalog_product_id": catalog_id,
            "product_id": item["product_id"],
            "quantity": item["quantity"],
            "is_primary": item.get("is_primary", False)
        })

    new_items = pd.DataFrame(items_list)

    # ===== INSERT (ATOMIC STYLE) =====
    catalog_df = pd.concat([catalog_df, new_catalog], ignore_index=True)
    items_df = pd.concat([items_df, new_items], ignore_index=True)

    return catalog_df, items_df

In [ ]:
catalog_product_df = create_catalog_products_df()
catalog_items_df = create_catalog_items_df()

data_no_bundle = ["helm hijab bogo", "accessories"]
data_bundle = ["helm bogo", "helm cakil"]

bundle_items = {
    "helm bogo": ["accessories"],
    "helm cakil": ["accessories"]
}

for category in data_no_bundle:
    subset = products_df[products_df["category"] == category]

    for _, row in subset.iterrows():
        data_catalog = {
            "product_name": f"{row['product_name']}",
            "is_bundle": False,
            "cost_price": row["cost_price"],
            "items": [
                {
                    "product_id": row["product_id"],
                    "quantity": 1,
                    "is_primary": True
                }
            ]
        }

        catalog_product_df, catalog_items_df = add_catalog_product(
            catalog_product_df,
            catalog_items_df,
            data_catalog, start_project
        )
for category in data_bundle:

    main_products = products_df[products_df["category"] == category]
    acc_products = products_df[products_df["category"] == "accessories"]

    for _, main in main_products.iterrows():
        for _, acc in acc_products.iterrows():

            data_catalog = {
                "product_name": f"{main['product_name']} + {acc['variant']}",
                "is_bundle": True,
                "cost_price": main["cost_price"] + acc["cost_price"],
                "items": [
                    {
                        "product_id": main["product_id"],
                        "quantity": 1,
                        "is_primary": True
                    },
                    {
                        "product_id": acc["product_id"],
                        "quantity": 1,
                        "is_primary": False
                    }
                ]
            }

            catalog_product_df, catalog_items_df = add_catalog_product(
                catalog_product_df,
                catalog_items_df,
                data_catalog,
                start_project
            )
catalog_product_df

/tmp/ipykernel_642/2114730350.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  catalog_df = pd.concat([catalog_df, new_catalog], ignore_index=True)


,catalog_product_id,product_name,is_bundle,cost_price,selling_price,tax,status,created_at,updated_at,created_by
0,CAT-3196-673-001,helm hijab bogo grey gloss,False,202000.0,282800.0,33936.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,CAT-3270-512-002,helm hijab bogo grey doff,False,202000.0,282800.0,33936.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,CAT-3349-444-003,helm hijab bogo hitam gloss,False,202000.0,282800.0,33936.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
3,CAT-2431-947-004,helm hijab bogo mint,False,202000.0,282800.0,33936.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
4,CAT-1969-671-005,helm hijab bogo sage,False,202000.0,282800.0,33936.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
...,...,...,...,...,...,...,...,...,...,...
155,CAT-2658-167-156,helm cakil pink soft + cembung smoke,True,215000.0,301000.0,36120.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
156,CAT-5666-631-157,helm cakil pink soft + flat clear,True,215000.0,301000.0,36120.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
157,CAT-2285-878-158,helm cakil pink soft + flat rainbow,True,215000.0,301000.0,36120.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
158,CAT-3742-431-159,helm cakil pink soft + pet doff,True,215000.0,301000.0,36120.0,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


In [ ]:
catalog_items_df

,catalog_product_id,product_id,quantity,is_primary
0,CAT-3196-673-001,PRD-6558-507-024,1,True
1,CAT-3270-512-002,PRD-3512-217-025,1,True
2,CAT-3349-444-003,PRD-7194-322-026,1,True
3,CAT-2431-947-004,PRD-2629-892-027,1,True
4,CAT-1969-671-005,PRD-8266-654-028,1,True
...,...,...,...,...
293,CAT-2285-878-158,PRD-8164-914-021,1,False
294,CAT-3742-431-159,PRD-4719-614-045,1,True
295,CAT-3742-431-159,PRD-5030-206-022,1,False
296,CAT-1173-543-160,PRD-4719-614-045,1,True


## Create Store

In [ ]:
def create_stores_df():
    return pd.DataFrame({
        "store_id": pd.Series(dtype="object"),
        "store_name": pd.Series(dtype="object"),
        "platform": pd.Series(dtype="object"),  # shopee / tiktok / dll
        "status": pd.Series(dtype="object"),    # active / inactive
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def add_store(df, data, time):

    required_fields = ["store_name", "platform"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    # detect multiple
    is_multiple = isinstance(data["store_name"], list)

    if is_multiple:
        new_df = pd.DataFrame(data)
        n = len(new_df)
    else:
        new_df = pd.DataFrame([data])
        n = 1

    # ===== AUTO FIELD =====
    new_df["status"] = "active"
    new_df["created_at"] = time
    new_df["updated_at"] = time

    # ===== GENERATE ID =====
    ids = create_code_id(df, "STR", "store_id", n)
    new_df["store_id"] = ids

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"
    new_df["created_by"] = admin_id

    # ===== INSERT =====
    df = pd.concat([df, new_df], ignore_index=True)

    return df

def add_store(df, data, time):

    required_fields = ["store_name", "platform"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    # detect multiple
    is_multiple = isinstance(data["store_name"], list)

    if is_multiple:
        new_df = pd.DataFrame(data)
        n = len(new_df)
    else:
        new_df = pd.DataFrame([data])
        n = 1

    # ===== AUTO FIELD =====
    new_df["status"] = "active"
    new_df["created_at"] = time
    new_df["updated_at"] = time

    # ===== GENERATE ID =====
    ids = create_code_id(df, "STR", "store_id", n)
    new_df["store_id"] = ids

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"
    new_df["created_by"] = admin_id

    # ===== INSERT =====
    df = pd.concat([df, new_df], ignore_index=True)

    return df

def deactivate_store(df, store_id, time):

    idx = df[df["store_id"] == store_id].index

    if len(idx) == 0:
        raise ValueError("Store tidak ditemukan")

    df.loc[idx, "status"] = "inactive"
    df.loc[idx, "updated_at"] = time

    return df



In [ ]:
stores_df = create_stores_df()

data = {
    "store_name" : ["Bogo Store", "Bogo Helmet", "Bogo Utama", "Bogo United"],
    "platform": ["tiktok", "shopee","tiktok", "shopee"],
}

stores_df = add_store(stores_df, data, start_project)
stores_df

/tmp/ipykernel_642/76108953.py:82: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_df], ignore_index=True)


,store_id,store_name,platform,status,created_at,updated_at,created_by
0,STR-2673-270-001,Bogo Store,tiktok,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,STR-4301-394-002,Bogo Helmet,shopee,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,STR-2839-533-003,Bogo Utama,tiktok,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
3,STR-8526-652-004,Bogo United,shopee,active,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


## Create Data PO

In [ ]:
def create_purchase_orders_df():
    return pd.DataFrame({
        "po_id": pd.Series(dtype="object"),
        "supplier_id": pd.Series(dtype="object"),
        "order_date": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),  # draft, ordered, received, completed
        "total_cost": pd.Series(dtype="float"),
        "created_at": pd.Series(dtype="object"),
        "updated_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_purchase_order_items_df():
    return pd.DataFrame({
        "po_item_id": pd.Series(dtype="object"),
        "po_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "quantity": pd.Series(dtype="int"),
        "cost_price": pd.Series(dtype="float"),
    })

def add_purchase_order(po_df, po_items_df, data, time):

    required_fields = ["supplier_id", "items"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")

    items = data["items"]

    if not isinstance(items, list) or len(items) == 0:
        raise ValueError("items harus list dan tidak boleh kosong")

    # ===== VALIDASI ITEMS =====
    for item in items:
        if "product_id" not in item or "quantity" not in item or "cost_price" not in item:
            raise ValueError("item harus punya product_id, quantity, cost_price")

    # ===== GENERATE PO ID =====
    po_id = create_code_id(po_df, "PO", "po_id")

    # ===== HITUNG TOTAL COST =====
    total_cost = sum(item["quantity"] * item["cost_price"] for item in items)

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"

    # ===== CREATE PO HEADER =====
    new_po = pd.DataFrame([{
        "po_id": po_id,
        "supplier_id": data["supplier_id"],
        "order_date": start_project,
        "status": "ordered",
        "total_cost": total_cost,
        "created_at": time,
        "updated_at": time,
        "created_by": admin_id
    }])

    # ===== CREATE PO ITEMS =====
    items_list = []

    for item in items:
        po_item_id = create_code_id(po_items_df, "POI", "po_item_id")

        items_list.append({
            "po_item_id": po_item_id,
            "po_id": po_id,
            "product_id": item["product_id"],
            "quantity": item["quantity"],
            "cost_price": item["cost_price"]
        })

    new_items = pd.DataFrame(items_list)

    # ===== INSERT =====
    po_df = pd.concat([po_df, new_po], ignore_index=True)
    po_items_df = pd.concat([po_items_df, new_items], ignore_index=True)

    return po_df, po_items_df

In [ ]:
po_df = create_purchase_orders_df()
po_items_df = create_purchase_order_items_df()

for _, row in suppliers_df.iterrows():

    # ambil semua produk dari supplier ini
    supplier_products = products_df[
        products_df["vendor_id"] == row["supplier_id"]
    ]
    # buat items list
    items = []

    for _, prod in supplier_products.iterrows():

        category = str(prod["category"]).lower()

        if category == "accessories":
            qty_acc = [500, 1000, 2000]
            qty = random.choice(qty_acc)

        else:  # primary product (helmet)
            qty_acc = [20, 50,100]
            qty = random.choice(qty_acc)

        items.append({
            "product_id": prod["product_id"],
            "quantity": qty,
            "cost_price": prod["cost_price"]
        })
    data = {
        "supplier_id": row["supplier_id"],
        "items": items
    }

    po_df, po_items_df = add_purchase_order(
        po_df,
        po_items_df,
        data, time=start_project
    )
po_df

/tmp/ipykernel_642/1088813010.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  po_df = pd.concat([po_df, new_po], ignore_index=True)


,po_id,supplier_id,order_date,status,total_cost,created_at,updated_at,created_by
0,PO-2826-234-001,SPLR-6274-501-001,2021-08-03 07:10:30,ordered,183350000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
1,PO-9645-291-002,SPLR-6208-341-002,2021-08-03 07:10:30,ordered,216000000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
2,PO-5728-214-003,SPLR-9649-612-003,2021-08-03 07:10:30,ordered,171700000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001
3,PO-3471-751-004,SPLR-6297-996-004,2021-08-03 07:10:30,ordered,43930000.0,2021-08-03 07:10:30,2021-08-03 07:10:30,EMP-1998-144-001


In [ ]:
po_items_df

,po_item_id,po_id,product_id,quantity,cost_price
0,POI-8085-469-001,PO-2826-234-001,PRD-8874-102-001,50,193000.0
1,POI-1190-474-001,PO-2826-234-001,PRD-5383-617-002,100,193000.0
2,POI-9069-216-001,PO-2826-234-001,PRD-2684-647-003,50,193000.0
3,POI-9609-625-001,PO-2826-234-001,PRD-9499-652-004,20,193000.0
4,POI-6808-360-001,PO-2826-234-001,PRD-7442-689-005,20,193000.0
5,POI-5833-858-001,PO-2826-234-001,PRD-7478-588-006,100,193000.0
6,POI-6035-631-001,PO-2826-234-001,PRD-4480-984-007,50,193000.0
7,POI-5918-961-001,PO-2826-234-001,PRD-3442-197-008,100,193000.0
8,POI-7841-163-001,PO-2826-234-001,PRD-6679-849-009,50,193000.0
9,POI-2862-709-001,PO-2826-234-001,PRD-2992-160-010,20,193000.0


## Create Transaksi Inventory

In [ ]:
def create_inventory_transaction_df():
    return pd.DataFrame({
        "transaction_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "date": pd.Series(dtype="object"),
        "quantity": pd.Series(dtype="int"),
        "type": pd.Series(dtype="object"),  # IN / OUT
        "actors_id": pd.Series(dtype="object"),
        "partners_id": pd.Series(dtype="object"),
        "explanation": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
        "reference": pd.Series(dtype="object"),
        "reference_id": pd.Series(dtype="object"),
    })

def create_current_stock_df():
    return pd.DataFrame({
        "product_id": products_df["product_id"],
        "stock": 0,
        "last_updated": start_project
    })

def update_stock(current_stock_df, product_id, qty, trx_type, time):

    idx = current_stock_df[current_stock_df["product_id"] == product_id].index

    if len(idx) == 0:
        stock = qty if trx_type == "IN" else -qty

        new_row = pd.DataFrame([{
            "product_id": product_id,
            "stock": stock,
            "last_updated": time
        }])

        current_stock_df = pd.concat([current_stock_df, new_row], ignore_index=True)

    else:
        if trx_type == "IN":
            current_stock_df.loc[idx, "stock"] += qty
        else:
            current_stock_df.loc[idx, "stock"] -= qty

        current_stock_df.loc[idx, "last_updated"] = time

    return current_stock_df

def add_inventory_transaction(trx_df, stock_df, data, time):

    required_fields = ["product_id", "quantity", "type", "partners_id",
                       "actors_id", "reference", "reference_id", "explanation"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing {field}")

    # 🔥 VALIDASI STOCK (untuk OUT)
    if data["type"] == "OUT":
        current = stock_df[stock_df["product_id"] == data["product_id"]]["stock"]

        if not current.empty:
            if current.iloc[0] < data["quantity"]:
                raise ValueError("Stock tidak cukup")

    # ===== CREATE TRANSACTION =====
    trx_id = create_code_id(trx_df, "TRX", "transaction_id")

    new_trx = pd.DataFrame([{
        "transaction_id": trx_id,
        "product_id": data["product_id"],
        "date": time,
        "quantity": data["quantity"],
        "type": data["type"],
        "actors_id": data.get("actors_id"),
        "partners_id": data.get("partners_id"),
        "explanation": data.get("explanation"),
        "created_at": time,
        "created_by": data.get("created_by", "SYSTEM"),
        "reference": data.get("reference"),
        "reference_id": data.get("reference_id")
    }])

    trx_df = pd.concat([trx_df, new_trx], ignore_index=True)

    # ===== UPDATE STOCK =====
    stock_df = update_stock(
        stock_df,
        data["product_id"],
        data["quantity"],
        data["type"], time
    )

    return trx_df, stock_df

In [ ]:
inventory_transaction_df = create_inventory_transaction_df()
current_stock_df = create_current_stock_df()

warehouse_emp = employees_df[employees_df["role"] == "warehouse"]["employee_id"].iloc[0]
admin_emp = employees_df[employees_df["role"] == "admin"]["employee_id"].iloc[0]

for _, row in po_items_df.iterrows():

    # ambil supplier dari po
    supplier_id = po_df.loc[
        po_df["po_id"] == row["po_id"], "supplier_id"
    ].iloc[0]

    data = {
        "product_id": row["product_id"],
        "quantity": row["quantity"],
        "type": "IN",
        "actors_id": warehouse_emp,
        "partners_id": supplier_id,
        "explanation": "Initial_stock",
        "created_by": admin_emp,
        "reference": "purchase",
        "reference_id": row["po_id"]
    }

    inventory_transaction_df, current_stock_df = add_inventory_transaction(
        inventory_transaction_df,
        current_stock_df,
        data, start_project
    )

inventory_transaction_df

/tmp/ipykernel_642/492158234.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  trx_df = pd.concat([trx_df, new_trx], ignore_index=True)


,transaction_id,product_id,date,quantity,type,actors_id,partners_id,explanation,created_at,created_by,reference,reference_id
0,TRX-8571-882-001,PRD-8874-102-001,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
1,TRX-5224-208-002,PRD-5383-617-002,2021-08-03 07:10:30,100,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
2,TRX-5629-903-003,PRD-2684-647-003,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
3,TRX-2322-258-004,PRD-9499-652-004,2021-08-03 07:10:30,20,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
4,TRX-9209-447-005,PRD-7442-689-005,2021-08-03 07:10:30,20,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
5,TRX-3987-679-006,PRD-7478-588-006,2021-08-03 07:10:30,100,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
6,TRX-8324-241-007,PRD-4480-984-007,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
7,TRX-2300-285-008,PRD-3442-197-008,2021-08-03 07:10:30,100,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
8,TRX-1698-775-009,PRD-6679-849-009,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001
9,TRX-6385-410-010,PRD-2992-160-010,2021-08-03 07:10:30,20,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001


In [ ]:
current_stock_df

,product_id,stock,last_updated
0,PRD-8874-102-001,50,2021-08-03 07:10:30
1,PRD-5383-617-002,100,2021-08-03 07:10:30
2,PRD-2684-647-003,50,2021-08-03 07:10:30
3,PRD-9499-652-004,20,2021-08-03 07:10:30
4,PRD-7442-689-005,20,2021-08-03 07:10:30
5,PRD-7478-588-006,100,2021-08-03 07:10:30
6,PRD-4480-984-007,50,2021-08-03 07:10:30
7,PRD-3442-197-008,100,2021-08-03 07:10:30
8,PRD-6679-849-009,50,2021-08-03 07:10:30
9,PRD-2992-160-010,20,2021-08-03 07:10:30


## Create Table Orders

In [ ]:
# add order, update order history

#status = ["progress", "shipped", "complete"]
def create_orders_df():
    return pd.DataFrame({
        "order_id": pd.Series(dtype="object"),
        "date": pd.Series(dtype="object"),
        "store_id": pd.Series(dtype="object"),
        "status_order": pd.Series(dtype="object"),
        "total_price": pd.Series(dtype="float"),
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_order_items_df():
    return pd.DataFrame({
        "order_details_id": pd.Series(dtype="object"),
        "order_id": pd.Series(dtype="object"),
        "catalog_product_id": pd.Series(dtype="object"),
        "quantity": pd.Series(dtype="int"),
        "price": pd.Series(dtype="float"),  # snapshot harga
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_order_status_history_df():
    return pd.DataFrame({
        "history_id": pd.Series(dtype="object"),
        "order_id": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),
        "timestamp": pd.Series(dtype="object"),
    })

def add_order(orders_df, order_items_df, history_df, catalog_df, data, time):

    required_fields = ["store_id", "items"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing {field}")

    items = data["items"]

    if not isinstance(items, list) or len(items) == 0:
        raise ValueError("items harus list & tidak boleh kosong")

    # ===== GENERATE ORDER ID =====
    order_id = create_code_id(orders_df, "ORD", "order_id")

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"

    # ===== PROCESS ITEMS =====
    items_list = []
    total_price = 0

    for item in items:
        if "catalog_product_id" not in item or "quantity" not in item:
            raise ValueError("item harus punya catalog_product_id & quantity")

        # ambil harga dari catalog
        catalog_row = catalog_df[
            catalog_df["catalog_product_id"] == item["catalog_product_id"]
        ]

        if catalog_row.empty:
            raise ValueError("catalog_product tidak ditemukan")

        price = catalog_row["selling_price"].iloc[0]

        total_price += price * item["quantity"]

        order_item_id = create_code_id(order_items_df, "ORIT", "order_details_id")

        items_list.append({
            "order_details_id": order_item_id,
            "order_id": order_id,
            "catalog_product_id": item["catalog_product_id"],
            "quantity": item["quantity"],
            "price": price,
            "created_at": time,
            "created_by": admin_id
        })

    new_items = pd.DataFrame(items_list)

    # ===== CREATE ORDER HEADER =====
    new_order = pd.DataFrame([{
        "order_id": order_id,
        "date": time,
        "store_id": data["store_id"],
        "status_order": "progress",
        "total_price": total_price,
        "created_at": time,
        "created_by": admin_id
    }])

    # ===== CREATE HISTORY =====
    new_history = pd.DataFrame([{
        "history_id": create_code_id(history_df, "HIS", "history_id"),
        "order_id": order_id,
        "status": "progress",
        "timestamp": time
    }])

    # ===== INSERT =====
    orders_df = pd.concat([orders_df, new_order], ignore_index=True)
    order_items_df = pd.concat([order_items_df, new_items], ignore_index=True)
    history_df = pd.concat([history_df, new_history], ignore_index=True)

    return orders_df, order_items_df, history_df

def update_order_status(orders_df, history_df, order_id, new_status, time):

    # ===== VALIDASI ORDER =====
    idx = orders_df[orders_df["order_id"] == order_id].index

    if len(idx) == 0:
        raise ValueError("Order tidak ditemukan")

    # ===== VALIDASI STATUS =====
    valid_status = ["progress", "shipped", "complete"]

    if new_status not in valid_status:
        raise ValueError(f"Status tidak valid: {new_status}")

    current_status = orders_df.loc[idx, "status_order"].iloc[0]

    # ===== OPTIONAL: CEK URUTAN STATUS 🔥 =====
    status_flow = {
        "progress": 1,
        "shipped": 2,
        "complete": 3
    }

    if status_flow[new_status] < status_flow[current_status]:
        raise ValueError("Tidak boleh mundur status")

    # ===== UPDATE ORDER =====
    orders_df.loc[idx, "status_order"] = new_status

    # ===== INSERT HISTORY =====
    new_history = pd.DataFrame([{
        "history_id": create_code_id(history_df, "HIS", "history_id"),
        "order_id": order_id,
        "status": new_status,
        "timestamp": time
    }])

    history_df = pd.concat([history_df, new_history], ignore_index=True)

    return orders_df, history_df

## Create Table Return Product

In [ ]:
# reason = ['buyer', 'seller']
# status = ["received", "inspected", "resolved"]
def create_returns_df():
    return pd.DataFrame({
        "return_id": pd.Series(dtype="object"),
        "order_id": pd.Series(dtype="object"),
        "return_date": pd.Series(dtype="object"),
        "reason": pd.Series(dtype="object"),  # buyer / seller
        "refund_amount": pd.Series(dtype="float"),
        "shipping_cost_borne_by": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),  # received, inspected, resolved
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_return_items_df():
    return pd.DataFrame({
        "return_item_id": pd.Series(dtype="object"),
        "return_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "qty": pd.Series(dtype="int"),
        "condition": pd.Series(dtype="object"),  # good / damaged
        "restockable": pd.Series(dtype="object"),  # yes / no
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def create_return_history_df():
    return pd.DataFrame({
        "history_id": pd.Series(dtype="object"),
        "return_id": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),
        "timestamp": pd.Series(dtype="object"),
    })

def add_return(returns_df, return_items_df, history_df, orders_df, data, time):

    required_fields = [
        "order_id",
        "reason",
        "items"
    ]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing {field}")

    # ===== VALIDASI ORDER =====
    order_row = orders_df[orders_df["order_id"] == data["order_id"]]

    if order_row.empty:
        raise ValueError("Order tidak ditemukan")

    if order_row["status_order"].iloc[0] != "complete":
        raise ValueError("Return hanya bisa untuk order yang sudah complete")

    # ===== VALIDASI ITEMS =====
    items = data["items"]

    if not isinstance(items, list) or len(items) == 0:
        raise ValueError("items harus list dan tidak boleh kosong")

    for item in items:
        if "product_id" not in item or "qty" not in item:
            raise ValueError("item harus punya product_id & qty")

    # ===== GENERATE RETURN ID =====
    return_id = create_code_id(returns_df, "RET", "return_id")

    # ===== CREATED BY =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    admin_id = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"

    # ===== HITUNG REFUND (SIMPLE) =====
    refund_amount = data.get("refund_amount", 0)

    # ===== CREATE RETURN HEADER =====
    new_return = pd.DataFrame([{
        "return_id": return_id,
        "order_id": data["order_id"],
        "return_date": time,
        "reason": data["reason"],
        "refund_amount": refund_amount,
        "shipping_cost_borne_by": data.get("shipping_cost_borne_by", "seller"),
        "status": "requested",
        "created_at": time,
        "created_by": admin_id
    }])

    # ===== CREATE RETURN ITEMS =====
    items_list = []

    for item in items:
        return_item_id = create_code_id(return_items_df, "RTI", "return_item_id")

        items_list.append({
            "return_item_id": return_item_id,
            "return_id": return_id,
            "product_id": item["product_id"],
            "qty": item["qty"],
            "condition": item.get("condition", "good"),
            "restockable": item.get("restockable", "yes"),
            "created_at": time,
            "created_by": admin_id
        })

    new_items = pd.DataFrame(items_list)

    # ===== CREATE HISTORY =====
    new_history = pd.DataFrame([{
        "history_id": create_code_id(history_df, "RHS", "history_id"),
        "return_id": return_id,
        "status": "requested",
        "timestamp": time
    }])

    # ===== INSERT =====
    returns_df = pd.concat([returns_df, new_return], ignore_index=True)
    return_items_df = pd.concat([return_items_df, new_items], ignore_index=True)
    history_df = pd.concat([history_df, new_history], ignore_index=True)

    return returns_df, return_items_df, history_df

def update_return_status(returns_df, history_df, return_id, new_status, time):

    # ===== VALIDASI RETURN =====
    idx = returns_df[returns_df["return_id"] == return_id].index

    if len(idx) == 0:
        raise ValueError("Return tidak ditemukan")

    # ===== VALIDASI STATUS =====
    valid_status = ["requested", "shipped", "received"]

    if new_status not in valid_status:
        raise ValueError(f"Status tidak valid: {new_status}")

    current_status = returns_df.loc[idx, "status"].iloc[0]

    # ===== VALIDASI FLOW (ANTI LONCAT / MUNDUR) 🔥 =====
    allowed_transition = {
        "requested": ["shipped"],
        "shipped": ["received"],
        "received": ["inspected"],
        "inspected": ["resolved"],
        "resolved": []
    }

    if new_status not in allowed_transition[current_status]:
        raise ValueError(f"Tidak boleh pindah dari {current_status} ke {new_status}")

    # ===== UPDATE STATUS =====
    returns_df.loc[idx, "status"] = new_status

    # ===== INSERT HISTORY =====
    new_history = pd.DataFrame([{
        "history_id": create_code_id(history_df, "RHS", "history_id"),
        "return_id": return_id,
        "status": new_status,
        "timestamp": time
    }])

    history_df = pd.concat([history_df, new_history], ignore_index=True)

    return returns_df, history_df

## Create Table Reject

In [ ]:
def create_reject_df():
    return pd.DataFrame({
        "reject_id": pd.Series(dtype="object"),
        "order_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "qc_id": pd.Series(dtype="object"),
        "quantity": pd.Series(dtype="int"),
        "reason": pd.Series(dtype="object"),
        "reject_type": pd.Series(dtype="object"),  # qc / supplier / return
        "status": pd.Series(dtype="object"),  # pending, sent_to_supplier, resolved
        "handled_by": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
    })

def create_reject_history_df():
    return pd.DataFrame({
        "reject_history_id": pd.Series(dtype="object"),
        "reject_id": pd.Series(dtype="object"),
        "status": pd.Series(dtype="object"),  # pending, returned_to_supplier, received_replacement
        "timestamp": pd.Series(dtype="object"),
        "note": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def add_reject(reject_df, qc_df, data, time):

    required_fields = [
        "order_id",
        "product_id",
        "qc_id",
        "quantity",
        "reason",
        "reject_type"
    ]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing {field}")

    # ===== VALIDASI QC =====
    qc_row = qc_df[qc_df["qc_id"] == data["qc_id"]]

    if qc_row.empty:
        raise ValueError("QC tidak ditemukan")

    if qc_row["status"].iloc[0] != "reject":
        raise ValueError("Reject hanya bisa dari QC yang status reject")

    # ===== HANDLER =====
    admin_df = employees_df[employees_df["role"] == "admin"]
    handler = admin_df["employee_id"].iloc[0] if not admin_df.empty else "SYSTEM"

    # ===== CREATE REJECT =====
    new_reject = pd.DataFrame([{
        "reject_id": create_code_id(reject_df, "RJT", "reject_id"),
        "order_id": data["order_id"],
        "product_id": data["product_id"],
        "qc_id": data["qc_id"],
        "quantity": data["quantity"],
        "reason": data["reason"],
        "reject_type": data["reject_type"],
        "status": "pending",
        "handled_by": handler,
        "created_at": time
    }])

    reject_df = pd.concat([reject_df, new_reject], ignore_index=True)

    return reject_df



## Create Packing QC

In [ ]:
def create_packing_qc_df():
    return pd.DataFrame({
        "qc_id": pd.Series(dtype="object"),
        "order_id": pd.Series(dtype="object"),
        "product_id": pd.Series(dtype="object"),
        "attempt_no": pd.Series(dtype="int"),
        "status": pd.Series(dtype="object"),  # pass / reject
        "reason": pd.Series(dtype="object"),
        "employee_id": pd.Series(dtype="object"),
        "date_qc": pd.Series(dtype="object"),
        "created_at": pd.Series(dtype="object"),
        "created_by": pd.Series(dtype="object"),
    })

def add_packing_qc(qc_df, data, time):

    required_fields = ["order_id", "product_id", "status", "employee_id"]

    for field in required_fields:
        if field not in data:
            raise ValueError(f"Missing {field}")

    # ===== VALIDASI STATUS =====
    if data["status"] not in ["pass", "reject"]:
        raise ValueError("status harus 'pass' atau 'reject'")

    # ===== CEK QC TERAKHIR 🔥 =====
    existing = qc_df[
        (qc_df["order_id"] == data["order_id"]) &
        (qc_df["product_id"] == data["product_id"])
    ]

    if not existing.empty:
        last_status = existing.sort_values("attempt_no").iloc[-1]["status"]

        # ❗ STOP kalau sudah PASS
        if last_status == "pass":
            raise ValueError("QC sudah PASS, tidak perlu QC lagi")

        attempt_no = existing["attempt_no"].max() + 1
    else:
        attempt_no = 1

    # ===== CREATE QC =====
    new_qc = pd.DataFrame([{
        "qc_id": create_code_id(qc_df, "QC", "qc_id"),
        "order_id": data["order_id"],
        "product_id": data["product_id"],
        "attempt_no": attempt_no,
        "status": data["status"],
        "reason": data.get("reason"),
        "employee_id": data["employee_id"],
        "date_qc": time,
        "created_at": time,
        "created_by": data["employee_id"]
    }])

    qc_df = pd.concat([qc_df, new_qc], ignore_index=True)

    return qc_df

def process_qc_and_reject(qc_df, reject_df, data, time):

    qc_df = add_packing_qc(qc_df, data, time)

    last_qc = qc_df.iloc[-1]

    # 🔥 AUTO REJECT
    if last_qc["status"] == "reject":
        reject_data = {
            "order_id": last_qc["order_id"],
            "product_id": last_qc["product_id"],
            "qc_id": last_qc["qc_id"],
            "quantity": 1,
            "reason": last_qc["reason"],
            "reject_type": "qc"
        }

        reject_df = add_reject(reject_df, qc_df, reject_data, time)

    return qc_df, reject_df

In [ ]:
import random
import numpy as np
from datetime import datetime, timedelta, date
from collections import defaultdict

random.seed(42)
np.random.seed(42)

# start_project = set_time(2023, 8, 3, 7, 10, 30)
SIM_START = date(2021, 8, 3)
SIM_END   = date(2025, 8, 3)
print(f"Simulasi: {SIM_START} → {SIM_END}  ({(SIM_END - SIM_START).days} hari)")

Simulasi: 2021-08-03 → 2025-08-03  (1461 hari)


### Event Calendar

In [ ]:
# ── Event Calendar ──────────────────────────────────────────
def build_double_dates(years):
    dd = set()
    for yr in years:
        for m in range(1, 13):
            try: dd.add(date(yr, m, m))
            except ValueError: pass
    return dd

DOUBLE_DATES = build_double_dates([2021, 2022, 2023, 2024, 2025])

# Lebaran Idul Fitri (±4 hari dari perkiraan)
LEBARAN_CENTERS = [date(2022, 5, 2), date(2023, 4, 22), date(2024, 4, 10), date(2025, 3, 31)]
LEBARAN_DATES   = set()
for _c in LEBARAN_CENTERS:
    for _d in range(-4, 6): LEBARAN_DATES.add(_c + timedelta(days=_d))

# Year-end (24 Des – 3 Jan)
YEAR_END_DATES = set()
for _yr in [2021, 2022, 2023, 2024, 2025]:
    for _d in range(0, 11): YEAR_END_DATES.add(date(_yr, 12, 24) + timedelta(days=_d))
    for _d in range(0, 4):  YEAR_END_DATES.add(date(_yr, 1, 1)  + timedelta(days=_d))

def classify_day(sim_date, months_elapsed):
    """Return (n_orders, event_type, phase)"""
    phase    = "early" if months_elapsed < 3 else "stable"
    is_leb   = sim_date in LEBARAN_DATES
    is_ye    = sim_date in YEAR_END_DATES
    near_dd  = any(abs((sim_date - dd).days) <= 2 for dd in DOUBLE_DATES)
    is_mon   = sim_date.weekday() == 0   # Senin

    if is_leb or is_ye:   event = "wow_holiday"
    elif near_dd:          event = "wow_double"
    elif is_mon:           event = "intermediate"
    else:                  event = "basic"

    _range = {
        ("wow_holiday", "early"):   (230, 250),
        ("wow_holiday", "stable"):  (300, 400),
        ("wow_double",  "early"):   (200, 240),
        ("wow_double",  "stable"):  (250, 350),
        ("intermediate","early"):   (150, 180),
        ("intermediate","stable"):  (180, 240),
        ("basic",       "early"):   (120, 130),
        ("basic",       "stable"):  (150, 220),
    }
    lo, hi = _range[(event, phase)]
    return random.randint(lo, hi), event, phase

print("Event calendar OK ✓")

Event calendar OK ✓


### Master Lookups & Stock Dict

In [ ]:
# ── Master Lookups ───────────────────────────────────────────
# Store weights: Bogo Store=tiktok(40%), Bogo Helmet=shopee(40%),
#                Bogo Utama=tiktok(10%), Bogo United=shopee(10%)
_store_ids = stores_df["store_id"].tolist()
_store_w   = [0.40, 0.40, 0.10, 0.10]

# Catalog per kategori
_primary = catalog_items_df[catalog_items_df["is_primary"] == True].copy()
_primary = _primary.merge(products_df[["product_id", "category"]], on="product_id")
_cat_map  = _primary.groupby("category")["catalog_product_id"].apply(list).to_dict()
_cat_order = ["helm bogo", "helm hijab bogo", "helm cakil", "accessories"]
_cat_w     = [0.50, 0.25, 0.10, 0.15]   # distribusi produk
_cat_arr   = [_cat_map.get(c, []) for c in _cat_order]

# Price & product lookup
_price_map = catalog_product_df.set_index("catalog_product_id")["selling_price"].to_dict()
_ci_map    = catalog_items_df.groupby("catalog_product_id")["product_id"].apply(list).to_dict()

# Stock dict (lebih cepat dari DataFrame lookup)
_stock         = current_stock_df.set_index("product_id")["stock"].to_dict()
_initial_stock = _stock.copy()

# Employee IDs
_admin_id = employees_df[employees_df["role"] == "admin"]["employee_id"].iloc[0]
_qc_ids   = employees_df[employees_df["role"] == "qc_packing"]["employee_id"].tolist()
_wh_ids   = employees_df[employees_df["role"] == "warehouse"]["employee_id"].tolist()

# Product → Supplier lookup (untuk reject return)
_product_supplier_map = products_df.set_index("product_id")["vendor_id"].to_dict()

print("Master lookups OK ✓")
print(f"  Catalog per kategori: { {k: len(v) for k, v in _cat_map.items()} }")

Master lookups OK ✓
  Catalog per kategori: {'accessories': 6, 'helm bogo': 102, 'helm cakil': 36, 'helm hijab bogo': 16}


In [ ]:
# ── Fast ID Generator ────────────────────────────────────────
_counters = defaultdict(int)

def gen_id(prefix: str) -> str:
    _counters[prefix] += 1
    n = _counters[prefix]
    return f"{prefix}-{random.randint(1000,9999)}-{random.randint(100,999)}-{n:04d}"

print("ID generator OK ✓")

ID generator OK ✓


### Init Result Tables & Buffers

In [ ]:
# ── Init empty result tables & buffers ──────────────────────
orders_df               = create_orders_df()
order_items_df          = create_order_items_df()
order_status_history_df = create_order_status_history_df()
packing_qc_df           = create_packing_qc_df()
reject_df               = create_reject_df()
returns_df              = create_returns_df()
return_items_df         = create_return_items_df()
return_history_df       = create_return_history_df()
reject_history_df       = create_reject_history_df()

# ── List buffers (batch-convert ke DataFrame setelah simulasi) ──
_buf_orders    = []; _buf_ord_items = []; _buf_history   = []
_buf_qc        = []; _buf_reject    = []; _buf_returns   = []
_buf_ret_items = []; _buf_ret_hist  = []; _buf_inv_trx   = []
_buf_po        = []; _buf_po_items  = []; _buf_reject_hist = []

# ── Status tracking (untuk update final saat flush) ──────────
_order_status_map = {}      # {order_id: status}
_return_status_map = {}     # {return_id: status}
_reject_status_map = {}     # {reject_id: status}
_ret_item_condition_map = {}  # {(return_id, product_id): (condition, restockable)}
_ret_item_id_map = {}

# ── Reject tracking ────────────────────────────────────────────
# Reject yang pending (belum dikirim ke supplier)
_pending_reject_ids_by_pid = defaultdict(list)
# Reject yang sudah dikirim ke supplier, menunggu replacement
_waiting_replacement_by_pid = defaultdict(list)
# Total qty replacement yang akan diterima di restock berikutnya
_pending_replacement_qty = defaultdict(int)

# ── Order lifecycle queues ────────────────────────────────────
_ship_queue     = defaultdict(list)  # {date: [(oid, pout, sid)]}
_complete_queue = defaultdict(list)  # {date: [(oid, pout, sid)]}

# ── Return sub-queues (3 tahap setelah order complete) ───────
_ret_req_queue  = defaultdict(list)  # {date: [(rid, oid, pout, sid)]}
_ret_ship_queue = defaultdict(list)  # {date: [rid]}
_ret_recv_queue = defaultdict(list)  # {date: [(rid, pout, sid)]}

# ── Restock set (reset tiap hari) ────────────────────────────
_restock_pids = set()

print("Tables & buffers siap ✓")

Tables & buffers siap ✓


### Main Loop  ⚡ (batch-mode, ~1 menit untuk 3 tahun)

In [ ]:
# ── Main Simulation Loop ─────────────────────────────────────
import sys, time as _time

_reject_reasons = [
    "produk rusak", "ada noda yg tidak bisa hilang",
    "Komponen produk rusak ", "part produk tidak berfungsi"
]

def _sample_qc_attempts():
    """Sampling berapa kali attempt QC (80%=1x, 10%=2x, 10%=5-6x)"""
    roll = random.random()
    if roll < 0.80:   return 1
    elif roll < 0.90: return 2
    else:             return random.choice([5, 6])

def _pick_catalog():
    ci   = random.choices(range(len(_cat_order)), weights=_cat_w, k=1)[0]
    cats = _cat_arr[ci]
    if not cats: cats = _cat_arr[0]
    return random.choice(cats)

def _check_restock(sim_dt, ts_day):
    """
    Eksekusi jam 17:00 agar tercatat SETELAH proses QC Packing (status pending aman).
    """
    ts_end_of_day = ts_day.replace(hour=17)
    by_supp = defaultdict(list)

    # Gabungkan semua PID yang punya kemungkinan restock/return/replace
    all_pids = set(_stock.keys()) | set(_pending_reject_ids_by_pid.keys()) | set(_pending_replacement_qty.keys())

    for pid in all_pids:
        curr = _stock.get(pid, 0)
        init = _initial_stock.get(pid, 500)

        is_low_stock = (curr < 0.30 * init) and (pid not in _restock_pids)
        reject_ids_to_send = _pending_reject_ids_by_pid.get(pid, []).copy()
        qty_replacement = _pending_replacement_qty.get(pid, 0)

        if is_low_stock or reject_ids_to_send or qty_replacement > 0:
            if is_low_stock:
                _restock_pids.add(pid)

            row = products_df[products_df["product_id"] == pid]
            if row.empty: continue

            supp_id = row["vendor_id"].iloc[0]
            cost    = row["cost_price"].iloc[0]

            by_supp[supp_id].append((
                pid, is_low_stock, reject_ids_to_send,
                qty_replacement, cost
            ))

            # KOSONGKAN antrean awal setelah dipindah ke by_supp agar tidak double counting
            if reject_ids_to_send:
                _pending_reject_ids_by_pid[pid].clear()
            if qty_replacement > 0:
                _pending_replacement_qty[pid] = 0

    for supp_id, items in by_supp.items():
        po_id = gen_id("PO")
        total_cost = 0
        has_in_transaction = False

        for pid, is_low_stock, reject_ids_send, qty_repl, cost in items:

            # ═════════════════════════════════════════════════════════
            # [1] RESTOCK NORMAL
            # ═════════════════════════════════════════════════════════
            if is_low_stock:
                row = products_df[products_df["product_id"] == pid]
                category = str(row["category"].iloc[0]).lower()
                restock_qty = random.choice([500, 1000, 2000]) if category == "accessories" else random.choice([20, 50, 100])

                total_cost += restock_qty * cost
                _buf_po_items.append({
                    "po_item_id": gen_id("POI"), "po_id": po_id,
                    "product_id": pid, "quantity": restock_qty, "cost_price": cost
                })
                _buf_inv_trx.append({
                    "transaction_id": gen_id("TRX"), "product_id": pid,
                    "date": ts_end_of_day, "quantity": restock_qty, "type": "IN",
                    "actors_id": _wh_ids[0] if _wh_ids else _admin_id,
                    "partners_id": supp_id, "explanation": "restock",
                    "created_at": ts_end_of_day, "created_by": _admin_id,
                    "reference": "purchase", "reference_id": po_id
                })
                _stock[pid] = _stock.get(pid, 0) + restock_qty
                has_in_transaction = True

            # ═════════════════════════════════════════════════════════
            # [2] TERIMA REPLACEMENT DARI SUPPLIER (Masuk Gudang)
            # ═════════════════════════════════════════════════════════
            # ═════════════════════════════════════════════════════════
            # [2] TERIMA REPLACEMENT DARI SUPPLIER (Masuk Gudang)
            # ═════════════════════════════════════════════════════════
            if qty_repl > 0:
                wait_list = _waiting_replacement_by_pid[pid]

                # Kita loop per ID reject yang diganti
                for rid in wait_list[:qty_repl]:

                    # 1. Catat Transaksi Inventory (IN qty 1 per reject_id)
                    _buf_inv_trx.append({
                        "transaction_id": gen_id("TRX"), "product_id": pid,
                        "date": ts_end_of_day, "quantity": 1, "type": "IN",
                        "actors_id": _wh_ids[0] if _wh_ids else _admin_id,
                        "partners_id": supp_id,
                        "explanation": "replacement_from_supplier",
                        "created_at": ts_end_of_day, "created_by": _admin_id,
                        "reference": "reject",     # <--- Referensinya adalah dokumen reject
                        "reference_id": rid        # <--- Diisi dengan ID Reject aslinya
                    })

                    # 2. Update Status Reject History
                    _buf_reject_hist.append({
                        "reject_history_id": gen_id("RJTH"),
                        "reject_id": rid, "status": "replacement_by_supplier",
                        "timestamp": ts_end_of_day,
                        "note": "Replacement diterima dari supplier",
                        "created_by": _admin_id
                    })
                    _reject_status_map[rid] = "replacement_by_supplier"

                # Update total stock gabungan
                _stock[pid] = _stock.get(pid, 0) + qty_repl

                # Potong waiting list yang sudah dipenuhi
                _waiting_replacement_by_pid[pid] = wait_list[qty_repl:]

            # ═════════════════════════════════════════════════════════
            # [3] KIRIM REJECT BARU KE SUPPLIER (Out)
            # ═════════════════════════════════════════════════════════
            if reject_ids_send:
                for rid in reject_ids_send:
                    _buf_reject_hist.append({
                        "reject_history_id": gen_id("RJTH"),
                        "reject_id": rid, "status": "returned_to_supplier",
                        "timestamp": ts_end_of_day,
                        "note": "Dikembalikan ke supplier untuk penggantian",
                        "created_by": _admin_id
                    })
                    _reject_status_map[rid] = "returned_to_supplier"

                _waiting_replacement_by_pid[pid].extend(reject_ids_send)
                # Jadwalkan agar besok diterima
                _pending_replacement_qty[pid] += len(reject_ids_send)

        # Buat PO Master jika ada transaksi IN
        if has_in_transaction:
            _buf_po.append({
                "po_id": po_id, "supplier_id": supp_id,
                "order_date": ts_end_of_day, "status": "received", "total_cost": total_cost,
                "created_at": ts_end_of_day, "updated_at": ts_end_of_day, "created_by": _admin_id
            })


total_days = (SIM_END - SIM_START).days
t0 = _time.time()
print(f"Memulai simulasi {total_days} hari (~{total_days/365:.1f} tahun)...")

for day_offset in range(total_days):
    sim_dt = SIM_START + timedelta(days=day_offset)
    ts_day = datetime.combine(sim_dt, datetime.min.time()).replace(hour=9)
    months_elapsed = (sim_dt.year - SIM_START.year) * 12 + \
                     (sim_dt.month - SIM_START.month)
    _restock_pids.clear()

    # ══════════════════════════════════════════════════════════════════
    # [A] Order SHIPPED (+1 hari setelah order pass QC)
    # ══════════════════════════════════════════════════════════════════
    if sim_dt in _ship_queue:
        for (oid, pout, sid) in _ship_queue[sim_dt]:
            ts_ship = ts_day.replace(hour=10)
            _order_status_map[oid] = "shipped"
            _buf_history.append({"history_id": gen_id("HIS"),
                "order_id": oid, "status": "shipped", "timestamp": ts_ship})
        del _ship_queue[sim_dt]

    # ══════════════════════════════════════════════════════════════════
    # [B] Order COMPLETED (+3 hari setelah order dibuat)
    #     inventory OUT | partner = store_id | reference = order_id
    # ══════════════════════════════════════════════════════════════════
    if sim_dt in _complete_queue:
        for (oid, pout, sid) in _complete_queue[sim_dt]:
            tc = ts_day.replace(hour=14)
            _order_status_map[oid] = "completed"
            _buf_history.append({"history_id": gen_id("HIS"),
                "order_id": oid, "status": "completed", "timestamp": tc})

            # 2% kemungkinan return
            if random.random() < 0.02:
                rid = gen_id("RET")

                refund = next(
                    row["total_price"]
                    for row in _buf_orders
                    if row["order_id"] == oid
                )

                reason = random.choice([
                    "produk tidak sesuai ukuran",
                    "produk tidak sesuai ukuran",
                    "produk rusak saat ekspedisi",
                    "produk rusak saat ekspedisi",
                    "warna tidak sesuai"
                ])

                _buf_returns.append({
                    "return_id": rid,
                    "order_id": oid,
                    "return_date": tc,
                    "reason": reason,
                    "refund_amount": refund,
                    "shipping_cost_borne_by": "seller",
                    "status": "requested",
                    "created_at": tc,
                    "created_by": _admin_id
                })

                _ret_req_queue[sim_dt + timedelta(days=1)].append(
                    (rid, oid, pout, sid)
                )
                _ret_ship_queue[sim_dt + timedelta(days=2)].append(rid)
                _ret_recv_queue[sim_dt + timedelta(days=3)].append(
                    (rid, pout, sid)
                )
        del _complete_queue[sim_dt]

    # ══════════════════════════════════════════════════════════════════
    # [C] Return Stage 1 – REQUESTED
    # ══════════════════════════════════════════════════════════════════
    if sim_dt in _ret_req_queue:
        for (rid, oid, pout, sid) in _ret_req_queue[sim_dt]:
            ts_req = ts_day.replace(hour=9)
            _buf_ret_hist.append({"history_id": gen_id("RHIS"),
                "return_id": rid, "status": "requested", "timestamp": ts_req})
            for pid, qty in pout:
                reti_id = gen_id("RETI")
                _ret_item_id_map[(rid, pid)] = reti_id  # <--- SIMPAN MAPPING KE DICT

                _buf_ret_items.append({"return_item_id": reti_id,
                    "return_id": rid, "product_id": pid, "qty": qty,
                    "condition": "pending", "restockable": "pending",
                    "created_at": ts_req, "created_by": _admin_id})
        del _ret_req_queue[sim_dt]

    # ══════════════════════════════════════════════════════════════════
    # [D] Return Stage 2 – SHIPPING (barang return dalam perjalanan)
    # ══════════════════════════════════════════════════════════════════
    if sim_dt in _ret_ship_queue:
        for rid in _ret_ship_queue[sim_dt]:
            ts_sret = ts_day.replace(hour=11)
            _buf_ret_hist.append({"history_id": gen_id("RHIS"),
                "return_id": rid, "status": "shipping", "timestamp": ts_sret})
            _return_status_map[rid] = "shipping"
        del _ret_ship_queue[sim_dt]

    # ══════════════════════════════════════════════════════════════════
    # [E] Return Stage 3 – RECEIVED
    #     80% restockable → inventory IN | partner = store | ref = return_id
    #     20% damaged → tidak ada transaksi inventory
    # ══════════════════════════════════════════════════════════════════
    if sim_dt in _ret_recv_queue:
        for (rid, pout, sid) in _ret_recv_queue[sim_dt]:
            ts_recv = ts_day.replace(hour=13)
            _buf_ret_hist.append({"history_id": gen_id("RHIS"),
                "return_id": rid, "status": "received", "timestamp": ts_recv})
            _return_status_map[rid] = "received"

            for pid, qty in pout:
                restockable = random.random() < 0.80
                _ret_item_condition_map[(rid, pid)] = (
                    "good" if restockable else "damaged",
                    "yes" if restockable else "no"
                )
                if restockable:
                    # <--- AMBIL return_item_id DARI MAPPING
                    reti_id = _ret_item_id_map.get((rid, pid))

                    _buf_inv_trx.append({"transaction_id": gen_id("TRX"),
                        "product_id": pid, "date": ts_recv, "quantity": qty,
                        "type": "IN",
                        "actors_id": _wh_ids[0] if _wh_ids else _admin_id,
                        "partners_id": sid, "explanation": "return",
                        "created_at": ts_recv, "created_by": _admin_id,
                        "reference": "return",
                        "reference_id": reti_id}) # <--- GUNAKAN return_item_id
                _stock[pid] = _stock.get(pid, 0) + qty
        del _ret_recv_queue[sim_dt]

    # ══════════════════════════════════════════════════════════════════
    # [F] Generate orders hari ini
    # ══════════════════════════════════════════════════════════════════
    n_orders, ev_type, phase = classify_day(sim_dt, months_elapsed)

    for _ in range(n_orders):
        store_id = random.choices(_store_ids, weights=_store_w, k=1)[0]
        n_items  = 1 if random.random() < 0.80 else 2
        chosen   = [_pick_catalog() for _ in range(n_items)]
        valid    = [c for c in chosen
                    if all(_stock.get(p, 0) >= 1 for p in _ci_map.get(c, []))]
        if not valid: continue

        oid   = gen_id("ORD")
        ts_o  = ts_day + timedelta(minutes=random.randint(0, 480))
        total = sum(_price_map.get(c, 0) for c in valid)

        _buf_orders.append({"order_id": oid, "date": ts_o,
            "store_id": store_id, "status_order": "progress",
            "total_price": total, "created_at": ts_o,
            "created_by": _admin_id})
        for c in valid:
            _buf_ord_items.append({"order_details_id": gen_id("ORIT"),
                "order_id": oid, "catalog_product_id": c,
                "quantity": 1, "price": _price_map.get(c, 0),
                "created_at": ts_o, "created_by": _admin_id})

        _order_status_map[oid] = "progress"
        _buf_history.append({"history_id": gen_id("HIS"),
            "order_id": oid, "status": "progress", "timestamp": ts_o})

        # ── [G] QC Packing (attempt-based untuk seluruh isi order) ──────────

        ts_qc  = ts_o + timedelta(hours=1)
        qc_emp = random.choice(_qc_ids) if _qc_ids else _admin_id

        # Ambil semua product dari seluruh catalog pada order ini
        all_products_for_qc = []

        for cat_id in valid:
            for pid in _ci_map.get(cat_id, []):
                all_products_for_qc.append(pid)

        # Product yang masih harus dicek sampai pass
        remaining_products = set(all_products_for_qc)

        attempt_no = 1

        while remaining_products and attempt_no <= 6:
            passed_this_round = []

            for pid in remaining_products:
                qc_id = gen_id("QC")

                # cek apakah product primary / accessory
                row = products_df[products_df["product_id"] == pid]
                if row.empty:
                    continue

                product_category = row["category"].iloc[0]
                supp_id = row["vendor_id"].iloc[0] # <--- AMBIL SUPPLIER ID DARI SINI

                is_accessory = (str(product_category).lower() == "accessories")

                # Probabilitas reject menurun tiap attempt
                if attempt_no == 1:
                    reject_prob = 0.30
                elif attempt_no == 2:
                    reject_prob = 0.15
                elif attempt_no == 3:
                    reject_prob = 0.07
                else:
                    reject_prob = 0.02
                # accessory maksimal 3 attempt
                if is_accessory and attempt_no >= 3:
                    is_reject = False
                else:
                    is_reject = random.random() < reject_prob

                qc_status = "reject" if is_reject else "pass"
                qc_reason = (
                    random.choice(_reject_reasons)
                    if is_reject
                    else "product baik"
                )

                ts_attempt = ts_qc + timedelta(
                    minutes=(attempt_no * 10)
                )

                # ========================
                # INSERT QC RECORD
                # ========================
                _buf_qc.append({
                    "qc_id": qc_id,
                    "order_id": oid,
                    "product_id": pid,
                    "attempt_no": attempt_no,
                    "status": qc_status,
                    "reason": qc_reason,
                    "employee_id": qc_emp,
                    "date_qc": ts_attempt,
                    "created_at": ts_attempt,
                    "created_by": qc_emp
                })

                # ========================
                # Jika REJECT
                # ========================
                if is_reject:

                    rjt_id = gen_id("RJT")

                    # Primary → pending
                    # Accessory → disposed
                    reject_status = (
                        "disposed"
                        if is_accessory
                        else "pending"
                    )

                    _buf_reject.append({
                        "reject_id": rjt_id,
                        "order_id": oid,
                        "product_id": pid,
                        "qc_id": qc_id,
                        "quantity": 1,
                        "reason": qc_reason,
                        "reject_type": "qc",
                        "status": reject_status,
                        "handled_by": _admin_id,
                        "created_at": ts_attempt
                    })
                    # stock langsung berkurang saat QC reject
                    _buf_inv_trx.append({
                        "transaction_id": gen_id("TRX"),
                        "product_id": pid,
                        "date": ts_attempt,
                        "quantity": 1,
                        "type": "OUT",
                        "actors_id": qc_emp,
                        "partners_id": supp_id,
                        "explanation": "qc_reject",
                        "created_at": ts_attempt,
                        "created_by": qc_emp,
                        "reference": "reject",
                        "reference_id": rjt_id
                    })

                    _stock[pid] = max(0, _stock.get(pid, 0) - 1)

                    _reject_status_map[rjt_id] = reject_status

                    # hanya yg bukan accesoris yg dikirim ke supplier
                    if not is_accessory:
                        # _pending_replacement_qty[pid] += 1
                        _pending_reject_ids_by_pid[pid].append(rjt_id)

                    _buf_reject_hist.append({
                        "reject_history_id": gen_id("RJTH"),
                        "reject_id": rjt_id,
                        "status": reject_status,
                        "timestamp": ts_attempt,
                        "note": f"QC reject attempt {attempt_no}: {qc_reason}",
                        "created_by": qc_emp
                    })

                # ========================
                # Jika PASS
                # ========================
                else:
                    passed_this_round.append(pid)

            # Hapus product yang sudah pass
            for pid in passed_this_round:
                remaining_products.discard(pid)

            attempt_no += 1
        # ── [H] Reserve stock + jadwalkan shipped (+1 hari) ──────

        pout = []

        for pid in all_products_for_qc:
            # hanya product yang akhirnya PASS yang dikirim
            if pid not in remaining_products:
                pout.append((pid, 1))

                # stock keluar untuk barang final yang lolos QC
                _buf_inv_trx.append({
                    "transaction_id": gen_id("TRX"),
                    "product_id": pid,
                    "date": ts_qc + timedelta(minutes=(attempt_no * 10)),
                    "quantity": 1,
                    "type": "OUT",
                    "actors_id": qc_emp,
                    "partners_id": store_id,
                    "explanation": "qc_pass_ready_to_ship",
                    "created_at": ts_qc,
                    "created_by": qc_emp,
                    "reference": "order",
                    "reference_id": oid
                })

                _stock[pid] = max(0, _stock.get(pid, 0) - 1)

        ship_date     = sim_dt + timedelta(days=1)
        complete_date = sim_dt + timedelta(days=3)

        if pout:
            _ship_queue[ship_date].append((oid, pout, store_id))
            _complete_queue[complete_date].append((oid, pout, store_id))

    # ── [I] Cek & Restock ────────────────────────────────────
    _check_restock(sim_dt, ts_day) # <--- Tambahkan parameter ts_day

    # ── Progress log setiap 90 hari ──────────────────────────
    if day_offset % 90 == 0:
        elapsed = _time.time() - t0
        eta = elapsed / max(day_offset, 1) * (total_days - day_offset)
        print(f"  [{sim_dt}] day {day_offset:4d}/{total_days}  "
              f"orders={len(_buf_orders):>7,}  "
              f"trx={len(_buf_inv_trx):>7,}  "
              f"ETA≈{eta/60:.1f}m", flush=True)

elapsed_total = _time.time() - t0
print(f"\nSimulasi selesai dalam {elapsed_total:.1f}s!")

Memulai simulasi 1461 hari (~4.0 tahun)...
  [2021-08-03] day    0/1461  orders=    130  trx=    370  ETA≈9.7m
  [2021-11-01] day   90/1461  orders= 13,231  trx= 39,518  ETA≈6.7m
  [2022-01-30] day  180/1461  orders= 32,887  trx= 97,850  ETA≈8.1m
  [2022-04-30] day  270/1461  orders= 52,294  trx=155,480  ETA≈8.2m
  [2022-07-29] day  360/1461  orders= 71,467  trx=212,530  ETA≈8.0m
  [2022-10-27] day  450/1461  orders= 90,092  trx=268,450  ETA≈7.7m
  [2023-01-25] day  540/1461  orders=109,869  trx=327,705  ETA≈7.2m
  [2023-04-25] day  630/1461  orders=129,784  trx=386,990  ETA≈6.8m
  [2023-07-24] day  720/1461  orders=148,644  trx=443,281  ETA≈6.2m
  [2023-10-22] day  810/1461  orders=167,226  trx=498,359  ETA≈5.6m
  [2024-01-20] day  900/1461  orders=186,582  trx=556,070  ETA≈5.0m
  [2024-04-19] day  990/1461  orders=206,843  trx=616,202  ETA≈4.3m
  [2024-07-18] day 1080/1461  orders=225,781  trx=672,532  ETA≈3.5m
  [2024-10-16] day 1170/1461  orders=244,814  trx=729,311  ETA≈2.8m
  [20

### Flush Buffer → DataFrames

In [ ]:
# ── Flush Buffer → DataFrames ────────────────────────────────
def _flush(buf, schema_fn):
    return pd.concat([schema_fn(), pd.DataFrame(buf)], ignore_index=True) if buf else schema_fn()

def _flush_append(buf, existing_df):
    if not buf:
        return existing_df
    return pd.concat([existing_df, pd.DataFrame(buf)], ignore_index=True)

inventory_transaction_df = _flush_append(_buf_inv_trx,   inventory_transaction_df)
orders_df               = _flush(_buf_orders,       create_orders_df)
order_items_df          = _flush(_buf_ord_items,    create_order_items_df)
order_status_history_df = _flush(_buf_history,      create_order_status_history_df)
packing_qc_df           = _flush(_buf_qc,           create_packing_qc_df)
reject_df               = _flush(_buf_reject,       create_reject_df)
reject_history_df       = _flush(_buf_reject_hist,  create_reject_history_df)
returns_df              = _flush(_buf_returns,      create_returns_df)
return_items_df         = _flush(_buf_ret_items,    create_return_items_df)
return_history_df       = _flush(_buf_ret_hist,     create_return_history_df)

po_df       = pd.concat([po_df,       pd.DataFrame(_buf_po)],
                         ignore_index=True) if _buf_po       else po_df
po_items_df = pd.concat([po_items_df, pd.DataFrame(_buf_po_items)],
                         ignore_index=True) if _buf_po_items else po_items_df

current_stock_df = pd.DataFrame([
    {"product_id": pid, "stock": stk,
     "last_updated": datetime.combine(SIM_END, datetime.min.time())}
    for pid, stk in _stock.items()
])

# ── Apply final status updates ke orders_df ──────────────────
if _order_status_map:
    _sm = pd.Series(_order_status_map, name="status_order")
    _sm.index.name = "order_id"
    orders_df = orders_df.set_index("order_id")
    orders_df.update(_sm.to_frame())
    orders_df = orders_df.reset_index()

# ── Apply final status updates ke returns_df ─────────────────
if _return_status_map:
    _rm = pd.Series(_return_status_map, name="status")
    _rm.index.name = "return_id"
    returns_df = returns_df.set_index("return_id")
    returns_df.update(_rm.to_frame())
    returns_df = returns_df.reset_index()

# ── Apply final status updates ke reject_df ──────────────────
if _reject_status_map:
    _rjm = pd.Series(_reject_status_map, name="status")
    _rjm.index.name = "reject_id"
    reject_df = reject_df.set_index("reject_id")
    reject_df.update(_rjm.to_frame())
    reject_df = reject_df.reset_index()

# ── Apply kondisi return_items (diisi setelah received) ──────
if _ret_item_condition_map:
    _cond_rows = [{"return_id": r, "product_id": p,
                   "condition": v[0], "restockable": v[1]}
                  for (r, p), v in _ret_item_condition_map.items()]
    _cond_df = pd.DataFrame(_cond_rows)
    if not _cond_df.empty:
        return_items_df = return_items_df.merge(
            _cond_df.rename(columns={
                "condition": "_cond", "restockable": "_rst"}),
            on=["return_id", "product_id"], how="left"
        )
        mask = return_items_df["_cond"].notna()
        return_items_df.loc[mask, "condition"]   = return_items_df.loc[mask, "_cond"]
        return_items_df.loc[mask, "restockable"] = return_items_df.loc[mask, "_rst"]
        return_items_df = return_items_df.drop(columns=["_cond", "_rst"])

print("=== HASIL SIMULASI ===")
rows = [
    ("orders",               orders_df),
    ("order_items",          order_items_df),
    ("order_status_history", order_status_history_df),
    ("packing_qc",           packing_qc_df),
    ("reject",               reject_df),
    ("reject_history",       reject_history_df),
    ("returns",              returns_df),
    ("return_items",         return_items_df),
    ("return_history",       return_history_df),
    ("inventory_transaction",inventory_transaction_df),
    ("purchase_orders",      po_df),
    ("po_items",             po_items_df),
    ("current_stock",        current_stock_df),
]
for name, df in rows:
    print(f"  {name:<25}: {len(df):>10,} rows")

# ── Validasi status distribution ─────────────────────────────
print("\nDistribusi status orders_df:")
print(orders_df["status_order"].value_counts().to_string())

print("\nDistribusi status returns_df:")
if len(returns_df):
    print(returns_df["status"].value_counts().to_string())

print("\nDistribusi kondisi return_items_df:")
if len(return_items_df):
    print(return_items_df[["condition","restockable"]].value_counts().to_string())

print("\nDistribusi status reject_df:")
if len(reject_df):
    print(reject_df["status"].value_counts().to_string())

print("\nDistribusi status reject_history_df:")
if len(reject_history_df):
    print(reject_history_df["status"].value_counts().to_string())

print(f"\nPending replacement (end of sim): {sum(_pending_replacement_qty.values()):,} units")

/tmp/ipykernel_642/2872786901.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([schema_fn(), pd.DataFrame(buf)], ignore_index=True) if buf else schema_fn()
/tmp/ipykernel_642/2872786901.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([schema_fn(), pd.DataFrame(buf)], ignore_index=True) if buf else schema_fn()
/tmp/ipykernel_642/2872786901.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future v

=== HASIL SIMULASI ===
  orders                   :    306,995 rows
  order_items              :    368,359 rows
  order_status_history     :    920,207 rows
  packing_qc               :    783,989 rows
  reject                   :    202,153 rows
  reject_history           :    419,347 rows
  returns                  :      6,146 rows
  return_items             :     11,899 rows
  return_history           :     18,412 rows
  inventory_transaction    :    914,860 rows
  purchase_orders          :      3,098 rows
  po_items                 :      5,809 rows
  current_stock            :         45 rows

Distribusi status orders_df:
status_order
completed    306411
shipped         390
progress        194

Distribusi status returns_df:
status
received     6132
requested       8
shipping        6

Distribusi kondisi return_items_df:
condition  restockable
good       yes            9523
damaged    no             2356
pending    pending          20

Distribusi status reject_df:
status
replace

In [ ]:
# # ── Flush Buffer → DataFrames ────────────────────────────────
# import pandas as pd

# def _flush(buf, schema_fn):
#     return pd.concat([schema_fn(), pd.DataFrame(buf)], ignore_index=True) if buf else schema_fn()

# def _flush_append(buf, existing_df):
#     if not buf:
#         return existing_df
#     return pd.concat([existing_df, pd.DataFrame(buf)], ignore_index=True)

# inventory_transaction_df = _flush_append(_buf_inv_trx,   inventory_transaction_df)
# orders_df               = _flush(_buf_orders,       create_orders_df)
# order_items_df          = _flush(_buf_ord_items,    create_order_items_df)
# order_status_history_df = _flush(_buf_history,      create_order_status_history_df)
# packing_qc_df           = _flush(_buf_qc,           create_packing_qc_df)
# reject_df               = _flush(_buf_reject,       create_reject_df)
# reject_history_df       = _flush(_buf_reject_hist,  create_reject_history_df)
# returns_df              = _flush(_buf_returns,      create_returns_df)
# return_items_df         = _flush(_buf_ret_items,    create_return_items_df)
# return_history_df       = _flush(_buf_ret_hist,     create_return_history_df)

# po_df       = pd.concat([po_df,       pd.DataFrame(_buf_po)],
#                          ignore_index=True) if _buf_po       else po_df
# po_items_df = pd.concat([po_items_df, pd.DataFrame(_buf_po_items)],
#                          ignore_index=True) if _buf_po_items else po_items_df

# current_stock_df = pd.DataFrame([
#     {"product_id": pid, "stock": stk,
#      "last_updated": datetime.combine(SIM_END, datetime.min.time())}
#     for pid, stk in _stock.items()
# ])

# # ── Apply final status updates ke orders_df ──────────────────
# if _order_status_map:
#     _sm = pd.Series(_order_status_map, name="status_order")
#     _sm.index.name = "order_id"
#     orders_df = orders_df.set_index("order_id")
#     orders_df.update(_sm.to_frame())
#     orders_df = orders_df.reset_index()

# # ── Apply final status updates ke returns_df ─────────────────
# if _return_status_map:
#     _rm = pd.Series(_return_status_map, name="status")
#     _rm.index.name = "return_id"
#     returns_df = returns_df.set_index("return_id")
#     returns_df.update(_rm.to_frame())
#     returns_df = returns_df.reset_index()

# # ── Apply final status updates ke reject_df ──────────────────
# if _reject_status_map:
#     _rjm = pd.Series(_reject_status_map, name="status")
#     _rjm.index.name = "reject_id"
#     reject_df = reject_df.set_index("reject_id")
#     reject_df.update(_rjm.to_frame())
#     reject_df = reject_df.reset_index()

# # ── Apply kondisi return_items (diisi setelah received) ──────
# if _ret_item_condition_map:
#     _cond_rows = [{"return_id": r, "product_id": p,
#                    "condition": v[0], "restockable": v[1]}
#                   for (r, p), v in _ret_item_condition_map.items()]
#     _cond_df = pd.DataFrame(_cond_rows)
#     if not _cond_df.empty:
#         return_items_df = return_items_df.merge(
#             _cond_df.rename(columns={
#                 "condition": "_cond", "restockable": "_rst"}),
#             on=["return_id", "product_id"], how="left"
#         )
#         mask = return_items_df["_cond"].notna()
#         return_items_df.loc[mask, "condition"]   = return_items_df.loc[mask, "_cond"]
#         return_items_df.loc[mask, "restockable"] = return_items_df.loc[mask, "_rst"]
#         return_items_df = return_items_df.drop(columns=["_cond", "_rst"])

# print("=== HASIL SIMULASI ===")
# rows = [
#     ("orders",               orders_df),
#     ("order_items",          order_items_df),
#     ("order_status_history", order_status_history_df),
#     ("packing_qc",           packing_qc_df),
#     ("reject",               reject_df),
#     ("reject_history",       reject_history_df),
#     ("returns",              returns_df),
#     ("return_items",         return_items_df),
#     ("return_history",       return_history_df),
#     ("inventory_transaction",inventory_transaction_df),
#     ("purchase_orders",      po_df),
#     ("po_items",             po_items_df),
#     ("current_stock",        current_stock_df),
# ]
# for name, df in rows:
#     print(f"  {name:<25}: {len(df):>10,} rows")

# # ── Validasi status distribution ─────────────────────────────
# print("\nDistribusi status orders_df:")
# print(orders_df["status_order"].value_counts().to_string())

# print("\nDistribusi status returns_df:")
# if len(returns_df):
#     print(returns_df["status"].value_counts().to_string())

# print("\nDistribusi kondisi return_items_df:")
# if len(return_items_df):
#     print(return_items_df[["condition","restockable"]].value_counts().to_string())

# print("\nDistribusi status reject_df:")
# if len(reject_df):
#     print(reject_df["status"].value_counts().to_string())

# print("\nDistribusi status reject_history_df:")
# if len(reject_history_df):
#     print(reject_history_df["status"].value_counts().to_string())

# print(f"\nPending replacement (end of sim): {sum(_pending_replacement_qty.values()):,} units")

### Quick Analytics

In [ ]:
# ── Quick Analytics ──────────────────────────────────────────
import pandas as pd

print("\n📊 RINGKASAN BISNIS 3 TAHUN")
print("=" * 55)

total_rev = orders_df["total_price"].sum()
n_days    = (SIM_END - SIM_START).days
print(f"Total Revenue      : Rp {total_rev:>18,.0f}")
print(f"Revenue/hari rata  : Rp {total_rev/n_days:>18,.0f}")
print(f"Total Orders       : {len(orders_df):>22,}")
print(f"Avg orders/hari    : {len(orders_df)/n_days:>22.1f}")

print("\nOrders per Platform:")
orders_df["platform"] = orders_df["store_id"].map(
    stores_df.set_index("store_id")["platform"])
print(orders_df.groupby("platform").agg(
    orders=("order_id","count"),
    revenue=("total_price","sum")
).to_string())

print("\nOrders per Store:")
orders_df["store_name"] = orders_df["store_id"].map(
    stores_df.set_index("store_id")["store_name"])
print(orders_df.groupby("store_name")["order_id"].count().to_string())

# Monthly revenue trend
orders_df["month"] = pd.to_datetime(orders_df["date"]).dt.to_period("M")
monthly = orders_df.groupby("month").agg(
    orders=("order_id","count"),
    revenue=("total_price","sum")
)
print(f"\nTrend 6 Bulan Terakhir:")
print(monthly.tail(6).to_string())

print(f"\nReturns            : {len(returns_df):>22,}  ({len(returns_df)/max(len(orders_df),1)*100:.2f}%)")
print(f"QC Rejects         : {len(reject_df):>22,}  ({len(reject_df)/max(len(packing_qc_df),1)*100:.2f}%)")
print(f"Restock POs        : {len(po_df)-4:>22,}")   # -4 karena 4 PO awal


📊 RINGKASAN BISNIS 3 TAHUN
Total Revenue      : Rp     94,861,011,000
Revenue/hari rata  : Rp         64,928,823
Total Orders       :                306,995
Avg orders/hari    :                  210.1

Orders per Platform:
          orders       revenue
platform                      
shopee    153591  4.747467e+10
tiktok    153404  4.738634e+10

Orders per Store:
store_name
Bogo Helmet    123057
Bogo Store     122704
Bogo United     30534
Bogo Utama      30700

Trend 6 Bulan Terakhir:
         orders       revenue
month                        
2025-03    7117  2.219937e+09
2025-04    6591  2.026786e+09
2025-05    6244  1.933355e+09
2025-06    6280  1.935748e+09
2025-07    6494  2.027395e+09
2025-08     405  1.239476e+08

Returns            :                  6,146  (2.00%)
QC Rejects         :                202,153  (25.79%)
Restock POs        :                  3,094


## Uji Coba

In [ ]:
# tables = {
#     "orders": orders_df, # bisa
#     "order_items": order_items_df,
#     "order_status_history": order_status_history_df,
#     "packing_qc": packing_qc_df,
#     "reject": reject_df,
#     "returns": returns_df,
#     "return_items": return_items_df,
#     "return_history": return_history_df,
#     "inventory_transaction": inventory_transaction_df,
#     "current_stock": current_stock_df,
#     "purchase_orders": po_df,
#     "po_items": po_items_df,
#     "stores": stores_df,
#     "products": products_df,
#     "catalog_items": catalog_items_df,
#     "catalog_product": catalog_product_df,
#     "employees": employees_df,
# }

In [ ]:
tables = {
    "1.orders": orders_df, # bisa
    "2. order_items": order_items_df,
    "3. order_status_history": order_status_history_df,
    "4. packing_qc": packing_qc_df,
    "5. reject": reject_df,
    "6. returns": returns_df,
    "7. return_items": return_items_df,
    "8. return_history": return_history_df,
    "9. inventory_transaction": inventory_transaction_df,
    "10. current_stock": current_stock_df,
    "11. purchase_orders": po_df,
    "12. po_items": po_items_df,
    "13. stores": stores_df,
    "14. products": products_df,
    "15. catalog_items": catalog_items_df,
    "16. catalog_product": catalog_product_df,
    "17. employees": employees_df,
    "18. reject_history" : reject_history_df,
    "19. suppliers" : suppliers_df
}
import os
path = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset_baru/"
for name, df in tables.items():
    df.to_csv(f"{path}{name}.csv", index=False)

In [ ]:
reject_df.product_id.value_counts()

,count
product_id,
PRD-5030-206-022,15735
PRD-6900-776-019,15666
PRD-8164-914-021,15581
PRD-8983-302-018,15558
PRD-8003-895-020,15546
PRD-9283-272-023,15430
PRD-5433-986-013,3889
PRD-2684-647-003,3858
PRD-4480-984-007,3837


In [ ]:
product_cacat = reject_df.product_id.unique()

for id_product in product_cacat:
    category = products_df[products_df.product_id == id_product].category.values[0]
    if category == "accessories":
        print(f"id product = {id_product}")
        order_id = reject_df[reject_df.product_id == id_product].order_id.values[0]
        catalog_id = order_items_df[order_items_df.order_id == order_id].catalog_product_id.values
        print(order_id)
        # print(catalog_product_df[catalog_product_df.catalog_product_id == catalog_id].product_name.values[0])


id product = PRD-8164-914-021
ORD-4598-801-0003
id product = PRD-8003-895-020
ORD-6539-121-0005
id product = PRD-5030-206-022
ORD-8711-327-0010
id product = PRD-9283-272-023
ORD-7211-793-0013
id product = PRD-6900-776-019
ORD-7565-603-0025
id product = PRD-8983-302-018
ORD-1815-790-0044


In [ ]:
packing_qc_df

,qc_id,order_id,product_id,attempt_no,status,reason,employee_id,date_qc,created_at,created_by
0,QC-2535-323-0001,ORD-2679-792-0001,PRD-2684-647-003,1,reject,produk rusak,EMP-9316-263-002,2021-08-03 16:29:00,2021-08-03 16:29:00,EMP-9316-263-002
1,QC-5557-928-0002,ORD-2679-792-0001,PRD-9283-272-023,1,pass,product baik,EMP-9316-263-002,2021-08-03 16:29:00,2021-08-03 16:29:00,EMP-9316-263-002
2,QC-3615-814-0003,ORD-2679-792-0001,PRD-2684-647-003,2,pass,product baik,EMP-9316-263-002,2021-08-03 16:39:00,2021-08-03 16:39:00,EMP-9316-263-002
3,QC-5803-949-0004,ORD-5333-926-0002,PRD-8003-895-020,1,pass,product baik,EMP-9316-263-002,2021-08-03 10:32:00,2021-08-03 10:32:00,EMP-9316-263-002
4,QC-6925-691-0005,ORD-5333-926-0002,PRD-3442-197-008,1,reject,produk rusak,EMP-9316-263-002,2021-08-03 10:32:00,2021-08-03 10:32:00,EMP-9316-263-002
...,...,...,...,...,...,...,...,...,...,...
783984,QC-7063-126-783985,ORD-6458-186-306993,PRD-8983-302-018,3,pass,product baik,EMP-9316-263-002,2025-08-02 14:55:00,2025-08-02 14:55:00,EMP-9316-263-002
783985,QC-7426-911-783986,ORD-5827-165-306994,PRD-9499-652-004,1,pass,product baik,EMP-9316-263-002,2025-08-02 17:57:00,2025-08-02 17:57:00,EMP-9316-263-002
783986,QC-9632-900-783987,ORD-5827-165-306994,PRD-9283-272-023,1,reject,part produk tidak berfungsi,EMP-9316-263-002,2025-08-02 17:57:00,2025-08-02 17:57:00,EMP-9316-263-002
783987,QC-3451-920-783988,ORD-5827-165-306994,PRD-9283-272-023,2,pass,product baik,EMP-9316-263-002,2025-08-02 18:07:00,2025-08-02 18:07:00,EMP-9316-263-002


In [ ]:
reject_df.status.value_counts()

,count
status,
replacement_by_supplier,108557
disposed,93516
returned_to_supplier,80


In [ ]:

qc_id = "ORD-7682-400-5943"
reject_df[reject_df.order_id == qc_id]

,reject_id,order_id,product_id,qc_id,quantity,reason,reject_type,status,handled_by,created_at


In [ ]:
current_stock_df

,product_id,stock,last_updated
0,PRD-8874-102-001,29,2025-08-03
1,PRD-5383-617-002,112,2025-08-03
2,PRD-2684-647-003,90,2025-08-03
3,PRD-9499-652-004,23,2025-08-03
4,PRD-7442-689-005,24,2025-08-03
5,PRD-7478-588-006,97,2025-08-03
6,PRD-4480-984-007,93,2025-08-03
7,PRD-3442-197-008,64,2025-08-03
8,PRD-6679-849-009,103,2025-08-03
9,PRD-2992-160-010,76,2025-08-03


In [ ]:
orders_df.store_name.value_counts()

,count
store_name,
Bogo Helmet,123057
Bogo Store,122704
Bogo Utama,30700
Bogo United,30534


In [ ]:
orders_df

,order_id,date,store_id,status_order,total_price,created_at,created_by,platform,store_name,month
0,ORD-2679-792-0001,2021-08-03 15:19:00,STR-2673-270-001,completed,303800.0,2021-08-03 15:19:00,EMP-1998-144-001,tiktok,Bogo Store,2021-08
1,ORD-5333-926-0002,2021-08-03 09:22:00,STR-2673-270-001,completed,303800.0,2021-08-03 09:22:00,EMP-1998-144-001,tiktok,Bogo Store,2021-08
2,ORD-4598-801-0003,2021-08-03 11:46:00,STR-4301-394-002,completed,303800.0,2021-08-03 11:46:00,EMP-1998-144-001,shopee,Bogo Helmet,2021-08
3,ORD-2827-400-0004,2021-08-03 12:42:00,STR-2839-533-003,completed,301000.0,2021-08-03 12:42:00,EMP-1998-144-001,tiktok,Bogo Utama,2021-08
4,ORD-6539-121-0005,2021-08-03 14:01:00,STR-2839-533-003,completed,303800.0,2021-08-03 14:01:00,EMP-1998-144-001,tiktok,Bogo Utama,2021-08
...,...,...,...,...,...,...,...,...,...,...
306990,ORD-3025-322-306991,2025-08-02 16:34:00,STR-2673-270-001,progress,33600.0,2025-08-02 16:34:00,EMP-1998-144-001,tiktok,Bogo Store,2025-08
306991,ORD-1528-139-306992,2025-08-02 10:29:00,STR-4301-394-002,progress,33600.0,2025-08-02 10:29:00,EMP-1998-144-001,shopee,Bogo Helmet,2025-08
306992,ORD-6458-186-306993,2025-08-02 13:25:00,STR-2673-270-001,progress,607600.0,2025-08-02 13:25:00,EMP-1998-144-001,tiktok,Bogo Store,2025-08
306993,ORD-5827-165-306994,2025-08-02 16:47:00,STR-4301-394-002,progress,303800.0,2025-08-02 16:47:00,EMP-1998-144-001,shopee,Bogo Helmet,2025-08


In [ ]:
reject_history_df

,reject_history_id,reject_id,status,timestamp,note,created_by
0,RJTH-4611-559-0001,RJT-4257-833-0001,pending,2021-08-03 16:29:00,QC reject attempt 1: produk rusak,EMP-9316-263-002
1,RJTH-5741-181-0002,RJT-1750-777-0002,pending,2021-08-03 10:32:00,QC reject attempt 1: produk rusak,EMP-9316-263-002
2,RJTH-7065-463-0003,RJT-8428-750-0003,pending,2021-08-03 10:42:00,QC reject attempt 2: Komponen produk rusak,EMP-9316-263-002
3,RJTH-8517-246-0004,RJT-4483-771-0004,pending,2021-08-03 12:56:00,QC reject attempt 1: Komponen produk rusak,EMP-9316-263-002
4,RJTH-4593-241-0005,RJT-8019-697-0005,disposed,2021-08-03 12:56:00,QC reject attempt 1: Komponen produk rusak,EMP-9316-263-002
...,...,...,...,...,...,...
419342,RJTH-5263-115-419343,RJT-1507-109-202113,returned_to_supplier,2025-08-02 17:00:00,Dikembalikan ke supplier untuk penggantian,EMP-1998-144-001
419343,RJTH-9552-175-419344,RJT-6134-997-201926,replacement_by_supplier,2025-08-02 17:00:00,Replacement diterima dari supplier,EMP-1998-144-001
419344,RJTH-9476-699-419345,RJT-3717-265-201928,replacement_by_supplier,2025-08-02 17:00:00,Replacement diterima dari supplier,EMP-1998-144-001
419345,RJTH-7290-343-419346,RJT-2048-623-202068,returned_to_supplier,2025-08-02 17:00:00,Dikembalikan ke supplier untuk penggantian,EMP-1998-144-001
